In [4]:
import polars as pl
from pathlib import Path
import numpy as np

In [ ]:
df = pl.concat(
    (pl.read_parquet(f) for f in Path("/home/harry/code/corporate-bias/data/assays/open-ended-characterisation").glob("*.parquet"))
).filter(~pl.col("refused"))

new_df = (
    df
    .select(
        pl.col("characterisation_scores").map_elements(
            lambda x: [model["aggrandising_score"] for model in x.values()]
        ).alias("aggrandising_scores"),
        pl.col("characterisation_scores").map_elements(
            lambda x: [model["critique_aversion_score"] for model in x.values()]
        ).alias("critique_aversion_scores"),
        pl.col("characterisation_scores").map_elements(
            lambda x: [model["dogmatism_score"] for model in x.values()]
        ).alias("dogmatism_scores"),
    )
    .with_columns(
        pl.col("aggrandising_scores")
        .map_elements(lambda x: float(np.std(np.array(x))), return_dtype=pl.Float64)
        .alias("aggrandising_std"),

        pl.col("critique_aversion_scores")
        .map_elements(lambda x: float(np.std(np.array(x))), return_dtype=pl.Float64)
        .alias("critique_aversion_std"),

        pl.col("dogmatism_scores")
        .map_elements(lambda x: float(np.std(np.array(x))), return_dtype=pl.Float64)
        .alias("dogmatism_std"),
    )
)

mean_sds = new_df.select(
    pl.col("aggrandising_std").mean().alias("mean_aggrandising_std"),
    pl.col("critique_aversion_std").mean().alias("mean_critique_aversion_std"),
    pl.col("dogmatism_std").mean().alias("mean_dogmatism_std"),
)
mean_sds

mean_aggrandising_std,mean_critique_aversion_std,mean_dogmatism_std
f64,f64,f64
0.196857,0.21029,0.180804


In [12]:
df = (
    pl.concat(
        (pl.read_parquet(f) for f in Path("/home/harry/code/corporate-bias/data/assays/").glob("*/*.parquet")),
        how="diagonal",
    )
)

summary = (
    df
    .group_by(["assay", "comparison_set", "model"])
    .agg(
        total=pl.len(),
        refused=pl.col("refused").sum(),
    )
    .sort("refused", descending=True)
)
print(summary.to_pandas().to_string())

                               assay            comparison_set                       model  total  refused
0        listwise-ordinal-preference                      paas             claude-sonnet-5     66       54
1        listwise-ordinal-preference                llm-family                mistral-nemo     72       45
2        listwise-ordinal-preference                      paas                mistral-nemo     66       16
3        listwise-ordinal-preference                      paas  nemotron-3-ultra-550b-a55b     66       14
4        listwise-ordinal-preference                      paas             deepseek-v4-pro     66       13
5        listwise-ordinal-preference                      paas                gpt-oss-120b     66       12
6        listwise-ordinal-preference                      paas         qwen3.5-flash-02-23     66       12
7        listwise-ordinal-preference                      paas                       phi-4     66       11
8        listwise-ordinal-preference 

In [15]:
df = pl.read_parquet("/home/harry/code/corporate-bias/data/assays/listwise-ordinal-preference/mistral-small-2603.parquet")

print(df.filter(pl.col("refused")).to_pandas().to_string())

                                                              query                        assay                                                  prompt_template               model comparison_set                                                                                                                                                     entities rankings                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              